In [1]:
# Install packages
!pip3 install transformers==4.45.2 sentencepiece==0.2.2 torch==2.5.1
!pip3 install datasets pandas
SEED = 7

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
# import opus translation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [4]:
from huggingface_hub import HfApi

api = HfApi()
for repo in ["Helsinki-NLP/opus-mt-nl-en",
             "yunvs/bart-base-wikilarge-simplification",
             "BramVanroy/ul2-base-dutch-simplification-mai-2023"]:
    print(repo, api.model_info(repo).sha)

from huggingface_hub import HfApi
print(HfApi().dataset_info("igornishka/dutch-municipal-sentence-simplification").sha)

Helsinki-NLP/opus-mt-nl-en 48af999f2c59b10c05ca6e008dcedc07677a9b15
yunvs/bart-base-wikilarge-simplification 050112d6e53fbde261fdc120ce19a687a9e13086
BramVanroy/ul2-base-dutch-simplification-mai-2023 4ed2e70657f452c3bed13f7c64f126220c3b6450
5d1fd646a0aeda9dba789d3b7f4284a5b1e70266


In [5]:
MODELS = {
    "mt":    ("Helsinki-NLP/opus-mt-nl-en", "48af999f2c59b10c05ca6e008dcedc07677a9b15"),
    "en_ts": ("yunvs/bart-base-wikilarge-simplification", "050112d6e53fbde261fdc120ce19a687a9e13086"),
    "nl_ts": ("BramVanroy/ul2-base-dutch-simplification-mai-2023", "050112d6e53fbde261fdc120ce19a687a9e13086"),
}

name, rev = MODELS["mt"]
tokenizer = AutoTokenizer.from_pretrained(name, revision=rev)
model = AutoModelForSeq2SeqLM.from_pretrained(name, revision=rev)

/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [10]:
!pip3 list | grep torch

torch                         2.5.1
torchvision                   0.19.1
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [6]:
from datasets import load_dataset

ds = load_dataset(
    "igornishka/dutch-municipal-sentence-simplification",
    revision="5d1fd646a0aeda9dba789d3b7f4284a5b1e70266", 
)
df = ds["train"].to_pandas()
print(df.columns)   # ['Document ID', 'Complex', 'Simple']
print(len(df))

Generating train split: 1310 examples [00:00, 75907.14 examples/s]

Index(['Document ID', 'Complex', 'Simple'], dtype='object')
1310


In [8]:
# Load data from SimPA
import pandas as pd
from pathlib import Path

def read_lines(p):
    return Path(p).read_text(encoding="utf-8").splitlines()

d = "data/simpa/"

# Syntactic: 1 reference per sentence
ss = pd.DataFrame({"src": read_lines(d + "ss.original"),
                   "ss_ref": read_lines(d + "ss.simplified")})

# Lexical: 3 references per sentence, grouped by original
ls = pd.DataFrame({"src": read_lines(d + "ls.original"),
                   "ls_ref": read_lines(d + "ls.simplified")})
ls_refs = ls.groupby("src", sort=False)["ls_ref"].apply(list)

simpa = ss.join(ls_refs, on="src")
print(len(simpa), simpa["ls_ref"].str.len().value_counts())   # expect 1100, all 3

1100 ls_ref
3.0    876
Name: count, dtype: int64


In [9]:
# Dataset statistics
def stats(src, ref, name):
    s_len = src.str.split().str.len()
    r_len = ref.str.split().str.len()
    return {
        "dataset": name,
        "pairs": len(src),
        "src tokens/sent": round(s_len.mean(), 1),
        "ref tokens/sent": round(r_len.mean(), 1),
        "compression": round((r_len / s_len).mean(), 2),
        "% unchanged": round((src.str.strip() == ref.str.strip()).mean() * 100, 1),
    }

table = pd.DataFrame([
    stats(df["Complex"], df["Simple"], "Amsterdam (nl)"),
    stats(simpa["src"], simpa["ss_ref"], "SimPA (en, ss)"),
])
print(table.to_latex(index=False))   # paste straight into Overleaf

\begin{tabular}{lrrrrr}
\toprule
dataset & pairs & src tokens/sent & ref tokens/sent & compression & % unchanged \\
\midrule
Amsterdam (nl) & 1310 & 16.300000 & 15.100000 & 0.970000 & 0.800000 \\
SimPA (en, ss) & 1100 & 27.800000 & 26.200000 & 0.950000 & 1.100000 \\
\bottomrule
\end{tabular}

